# Chapter 51 — Vector Search Application

Build a complete document search engine
This standalone chapter combines document storage, embeddings, FAISS retrieval, metadata filtering, and readable search results. It does not depend on earlier notebooks. The sample financial-crime documents are synthetic learning examples.

# 1. Install dependencies

In [ ]:
%pip install -q sentence-transformers faiss-cpu numpy pandas


# 2. Imports

In [3]:
from typing import Optional
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

/Users/vimal/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [4]:
print("FAISS loaded:", hasattr(faiss, "IndexFlatIP"))


FAISS loaded: True


# 3. Create a document collection

In [5]:
documents = [
    {"id":"doc_001","text":"A bank may review international transfers inconsistent with a customer's expected activity.","topic":"cross_border","source":"monitoring_guide"},
    {"id":"doc_002","text":"Structuring involves splitting transactions into smaller amounts to evade reporting or monitoring controls.","topic":"structuring","source":"aml_glossary"},
    {"id":"doc_003","text":"Sanctions screening checks customers and payment parties against applicable sanctions lists.","topic":"sanctions","source":"screening_guide"},
    {"id":"doc_004","text":"Entity resolution helps determine whether two records refer to the same person or organization.","topic":"entity_resolution","source":"data_quality_guide"},
    {"id":"doc_005","text":"A false positive is an alert investigated and found not to represent suspicious activity.","topic":"alert_quality","source":"monitoring_guide"},
    {"id":"doc_006","text":"Know Your Customer procedures verify customer identity and establish expected account activity.","topic":"kyc","source":"customer_due_diligence"},
    {"id":"doc_007","text":"Trade-based money laundering can misuse invoices, goods, or trade transactions to move illicit value.","topic":"trade_based_ml","source":"aml_glossary"},
    {"id":"doc_008","text":"Investigators should document evidence, analysis, and rationale supporting an alert decision.","topic":"investigation","source":"case_handling_guide"},
    {"id":"doc_009","text":"A sudden increase in cross-border payment frequency may require comparison with historical customer behavior.","topic":"cross_border","source":"monitoring_guide"},
    {"id":"doc_010","text":"A transaction monitoring system applies rules or models to identify activity needing investigation.","topic":"monitoring","source":"monitoring_guide"},
    {"id":"doc_011","text":"Correspondent banking allows one financial institution to provide services to another institution.","topic":"correspondent_banking","source":"payments_guide"},
    {"id":"doc_012","text":"Unusual payment corridors can be one factor when assessing risk of an international transfer.","topic":"cross_border","source":"monitoring_guide"},
    {"id":"doc_013","text":"Metadata filters can restrict search results to a topic, source, department, or document type.","topic":"search_engine","source":"vector_search_notes"},
    {"id":"doc_014","text":"Cosine similarity compares the direction of two vectors and is commonly used for semantic retrieval.","topic":"vector_search","source":"vector_search_notes"},
    {"id":"doc_015","text":"A retrieval system should return source text so users can inspect evidence behind a result.","topic":"search_engine","source":"vector_search_notes"}
]
documents_df = pd.DataFrame(documents)


In [6]:
documents_df.head(1)

,id,text,topic,source
0,doc_001,A bank may review international transfers inco...,cross_border,monitoring_guide


# 4. Generate document embeddings

In [9]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
texts = [item["text"] for item in documents]

In [12]:
texts

["A bank may review international transfers inconsistent with a customer's expected activity.",
 'Structuring involves splitting transactions into smaller amounts to evade reporting or monitoring controls.',
 'Sanctions screening checks customers and payment parties against applicable sanctions lists.',
 'Entity resolution helps determine whether two records refer to the same person or organization.',
 'A false positive is an alert investigated and found not to represent suspicious activity.',
 'Know Your Customer procedures verify customer identity and establish expected account activity.',
 'Trade-based money laundering can misuse invoices, goods, or trade transactions to move illicit value.',
 'Investigators should document evidence, analysis, and rationale supporting an alert decision.',
 'A sudden increase in cross-border payment frequency may require comparison with historical customer behavior.',
 'A transaction monitoring system applies rules or models to identify activity need

In [56]:
document_embeddings = model.encode(
    texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True
).astype("float32")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [57]:
documents_embeddings

array([[ 0.01895588, -0.04373493, -0.05704002, ..., -0.04006692,
        -0.0237683 , -0.00620279],
       [ 0.02967618,  0.04703083, -0.08784763, ..., -0.03664692,
         0.00702433, -0.00860999],
       [-0.0459016 ,  0.05422967, -0.01667891, ...,  0.02195304,
         0.11224865, -0.12726276],
       ...,
       [ 0.02671731,  0.06356149, -0.01615529, ..., -0.02299431,
         0.10316034,  0.06312059],
       [-0.00863723,  0.00408157, -0.07649782, ...,  0.08541644,
         0.03387145, -0.02226307],
       [-0.06924345,  0.0689934 , -0.07812959, ...,  0.07969987,
         0.04518694,  0.02453122]], dtype=float32)

In [58]:
document_embeddings.shape

(15, 384)

In [59]:
assert documents_embeddings.shape[0] == len(documents)


In [60]:
assert documents_embeddings.ndim == 2


# 5. Build the FAISS index

In [61]:
dimension = documents_embeddings.shape[1]
dimension

384

In [62]:
vector_index = faiss.IndexFlatIP(dimension)


In [63]:
vector_index

<faiss.swigfaiss.IndexFlatIP; proxy of <Swig Object of type 'faiss::IndexFlatIP *' at 0x146f46e40> >

In [64]:
vector_index.add(documents_embeddings)


In [65]:
# Dimensions
vector_index.d

384

In [66]:
# Vector Indexed
vector_index.ntotal

15

In [67]:
assert vector_index.ntotal == len(documents)


# 6. Build The reusable seach engine 

In [73]:
def search_documents(query :str, k:int =5, topic:Optional[str] = None) -> list[dict]:
    if not isinstance(query, str) or not query.strip():
        raise ValueError("query must be a non-empty string")
    if not isinstance(k, int) or not 1 <= k <= len(documents):
        raise ValueError(f"k must be between 1 and {len(documents)}")
    if topic is not None and not isinstance(topic, str):
        raise ValueError("topic must be a string or None")

    query_embedding = model.encode([query],convert_to_numpy=True,normalize_embeddings=True).astype("float32")
    target_k = len(documents) if topic is not None else k
    
    scores, positions = vector_index.search(query_embedding, len(documents))    
    results = []

    for score, position in zip(scores[0], positions[0]):
        if position == -1:
            continue

        position = int(position)
        record = documents[position]

        results.append({"id" : record["id"],
                        "text": record["text"],
                        "topic": record["topic"],
            "source": record["source"],
            "similarity": float(score)})
        if len(results) == k:
            break
    return results


In [74]:
query = "How can a bank find risky overseas payments?"
results = search_documents(query, k =3)

In [75]:
results

[{'id': 'doc_012',
  'text': 'Unusual payment corridors can be one factor when assessing risk of an international transfer.',
  'topic': 'cross_border',
  'source': 'monitoring_guide',
  'similarity': 0.6586640477180481},
 {'id': 'doc_001',
  'text': "A bank may review international transfers inconsistent with a customer's expected activity.",
  'topic': 'cross_border',
  'source': 'monitoring_guide',
  'similarity': 0.46763473749160767},
 {'id': 'doc_003',
  'text': 'Sanctions screening checks customers and payment parties against applicable sanctions lists.',
  'topic': 'sanctions',
  'source': 'screening_guide',
  'similarity': 0.4090046286582947}]

In [78]:
for rank, result in enumerate(results, start = 1):
    print(f"\\nRank {rank} | {result['id']} | topic={result['topic']} | source={result['source']}")
    print(f"Similarity: {result['similarity']:.4f}")
    print(result["text"])

\nRank 1 | doc_012 | topic=cross_border | source=monitoring_guide
Similarity: 0.6587
Unusual payment corridors can be one factor when assessing risk of an international transfer.
\nRank 2 | doc_001 | topic=cross_border | source=monitoring_guide
Similarity: 0.4676
A bank may review international transfers inconsistent with a customer's expected activity.
\nRank 3 | doc_003 | topic=sanctions | source=screening_guide
Similarity: 0.4090
Sanctions screening checks customers and payment parties against applicable sanctions lists.
